Think of your dataset as raw ingredients delivered to a kitchen. Some eggs are cracked, some labels are missing, some units don't match your recipe. Data wrangling is the prep work, ie, washing, peeling, chopping, before you can actually cook (analyze or model). Skip it, and even the best recipe (algorithm) produces a bad meal. That's the "garbage in, garbage out" rule.

Real-world example-

We will use Nigerian unemployment survey data which has 5,000 rows, 8 columns (Age, Gender, Region, Location, Education_Level, Employment_Status, Years_Of_Experience, Monthly_Income_NGN). Real surveys always have rot: a field worker skips a question, a form gets damaged, someone refuses to answer. So this dataset has missing values scattered across Region, Education_Level, Years_Of_Experience, and Monthly_Income_NGN.

In [15]:
#we import our libraries

import pandas as pd
import numpy as np

In [16]:
#we load our dataset of interest
#display first 10 and last 10 rows of the dataset

nigeria_unemployment_data = pd.read_csv('nigeria_unemployment_missing_data.csv')
nigeria_unemployment_data.head(10)

,Age,Gender,Region,Location,Education_Level,Employment_Status,Years_Of_Experience,Monthly_Income_NGN
0,53,Male,North West,Urban,NaN,Employed,37.0,84220.84
1,43,Female,South West,Urban,Primary,Employed,24.0,69911.72
2,29,Female,South South,Rural,NaN,Underemployed,NaN,31096.21
3,57,Female,South West,Rural,Primary,Employed,42.0,108994.91
4,22,Female,South South,Rural,NaN,Underemployed,5.0,31807.56
5,35,Male,South East,Urban,Secondary,Employed,17.0,NaN
6,53,Female,North West,Urban,Primary,Underemployed,35.0,31198.39
7,33,Male,North West,Rural,NaN,Employed,18.0,152089.71
8,37,Male,North East,Urban,Primary,Employed,21.0,118601.56
9,25,Female,North West,Urban,Primary,Underemployed,10.0,38302.58


In [17]:
nigeria_unemployment_data.tail(10)

,Age,Gender,Region,Location,Education_Level,Employment_Status,Years_Of_Experience,Monthly_Income_NGN
4990,43,Male,North Central,Rural,Primary,Employed,27.0,120888.72
4991,61,Female,NaN,Urban,Secondary,Employed,46.0,114676.57
4992,51,Male,South West,Urban,Tertiary,Employed,29.0,159731.51
4993,37,Female,South South,Urban,Primary,Underemployed,20.0,23087.41
4994,27,Female,North West,Urban,NaN,Underemployed,11.0,NaN
4995,46,Female,North West,Urban,Secondary,Underemployed,30.0,46905.52
4996,60,Female,South East,Rural,NaN,Underemployed,43.0,NaN
4997,61,Female,South West,Rural,Secondary,Underemployed,46.0,31289.56
4998,62,Female,North West,Urban,NaN,Underemployed,45.0,30686.16
4999,37,Male,North Central,Urban,NaN,Underemployed,20.0,26491.24


In [18]:
#check for the shape of the dataset
nigeria_unemployment_data.shape

#the dataset has 5000 rows and 8 columns

(5000, 8)

In [19]:
#Let's get a quick statistical summary of the dataset
nigeria_unemployment_data.describe()

,Age,Years_Of_Experience,Monthly_Income_NGN
count,5000.000000,4500.000000,4592.000000
mean,39.666600,21.835333,58114.540209
std,14.384312,14.280826,50779.924493
min,15.000000,0.000000,0.000000
25%,27.000000,9.000000,21051.712500
50%,40.000000,22.000000,42908.875000
75%,52.000000,34.000000,93927.700000
max,64.000000,49.000000,261319.160000


In [20]:
#check for missing values in the dataset

nigeria_unemployment_data.isnull().sum()

Age                       0
Gender                    0
Region                  250
Location                  0
Education_Level        1127
Employment_Status         0
Years_Of_Experience     500
Monthly_Income_NGN      408
dtype: int64

Education_Level is missing in 1,127 of 5,000 rows.
Option A — Drop it: dropna(inplace=True) deletes every row with any missing value. This will shrink the dataset from 5,000 rows to 3,018 rows. that's a 40% loss. This is the blunt-instrument approach: simple, but wasteful if missingness isn't random.

Option B — Fill numeric gaps: For Years_Of_Experience, the notebook will compute the column mean (~21.8 years) and use fillna(average_experience) to plug the gaps rather than throw away good data in other columns.

Option C — Fill categorical gaps with a label: For Education_Level, instead of guessing "Primary" vs "Secondary" (which would fabricate a fact), the notebook will fill missing entries with the honest label 'Unknown', preserving that the data was genuinely absent rather than pretending it wasn't.
The key judgment call this notebook will teach: don't apply one strategy blindly to every column. Dropping is right when missingness is rare; filling with a statistic (mean) suits numeric data; filling with "Unknown" suits categories where fabricating a value would be dishonest.

In [21]:
#check for missing values in the monthly_income column

nigeria_unemployment_data['Monthly_Income_NGN'].isnull().sum()

np.int64(408)

there are 408 missing values in monthly income. should we drop them, fill them with mean or median?
First, let's check whether income is skewed

In [22]:
#let's check whether income is skewed

nigeria_unemployment_data['Monthly_Income_NGN'].skew()

np.float64(0.8105241636252166)

A skew of 0.81 is positive, which means it's right-skewed, a long tail stretching toward higher incomes. Most people cluster at lower-to-mid income, but a smaller number of high earners pull the tail (and the mean) upward.
As a rough rule of thumb, skew between -0.5 and 0.5 is roughly symmetric, and anything past ±0.5 (like your 0.81) is "moderately skewed." So this isn't wildly extreme, but it's enough to matter.
Why this matters for filling missing values: in a right-skewed distribution, the mean gets dragged upward by those high earners. it no longer represents a "typical" person's income. The median sits at the actual midpoint of the data and isn't distorted by outliers, so it's the more honest choice for filling Monthly_Income_NGN.

In [23]:
#Let's compare the mean and median of the monthly income column to determine if the data is skewed
mean_income = nigeria_unemployment_data['Monthly_Income_NGN'].mean()
median_income = nigeria_unemployment_data['Monthly_Income_NGN'].median()
print(mean_income, median_income)

58114.54020905924 42908.875


That gap tells the whole story: mean ≈ ₦58,114.54 vs median ≈ ₦42,908.88. the mean is about ₦15,200 higher (roughly 35% higher than the median). 
That's a big pull, confirming what the skew of 0.81 predicted: a chunk of high earners is dragging the mean well above what a "typical" person actually earns.
If we fill the missing 408 values with the mean, we'd be quietly inserting ₦58k as "normal" income into over 400 rows, inflating the dataset's picture of typical earnings. Filling with the median keeps those rows closer to what most people in the sample actually make.

In [24]:
#Let's fill the 408 missing values in the monthly income column with the median value of the column

median_income_value = nigeria_unemployment_data['Monthly_Income_NGN'].median()    
nigeria_unemployment_data['Monthly_Income_NGN'] =  (nigeria_unemployment_data['Monthly_Income_NGN'].fillna(median_income_value)) 

#the code means- run through the monthly income of the dataset, calculate the median and store in a temporary variable called median_income_value. 
#Then, fill the missing values in the monthly income column with the median value stored in the temporary variable, and return the updated column.

In [25]:
#Let's confirm that the missing values have been filled in the monthly income column
nigeria_unemployment_data['Monthly_Income_NGN'].isnull().sum()

np.int64(0)

In [29]:
#Now let's fill in the missing values in the years_of_experience column with the mean value of the column


mean_experience_value = nigeria_unemployment_data['Years_Of_Experience'].mean()
nigeria_unemployment_data['Years_Of_Experience'] = nigeria_unemployment_data['Years_Of_Experience'].fillna(mean_experience_value)

In [ ]:
#Let's confirm that the Years_Of_Experience column has no missing values anymore

nigeria_unemployment_data['Years_Of_Experience'].isnull().sum()

np.int64(0)

In [ ]:
#Let's check how many missing values under education level column

nigeria_unemployment_data['Education_Level'].isnull().sum()

np.int64(1127)

In [32]:
#Let's fill the missing values under the education level column with the mode of the column (mode is the most frequently occurring value in a column)


mode_education_level = nigeria_unemployment_data['Education_Level'].mode()[0]  
    #####run through the education level column, calculate the mode and store in a temporary variable called mode_education_level.
nigeria_unemployment_data['Education_Level'] = nigeria_unemployment_data['Education_Level'].fillna(mode_education_level)
######fill the missing values in the education level column with the mode value stored in the temporary variable, and return the updated column.

while the method of using mode to fill missing values in a non-numerical variable is common in statistics and modelling, it is sometimes not safer.
when doing analysis where transparency about data quality matters, its safer to use 'Unkown' to fill in the missing values. the code for this is;

nigeria_unemployment_data['Education_Level'] = nigeria_unemployment_data['Education_Level'].fillna('Unknown')

In [33]:
#lets check if the missing values in the education level column have been filled
nigeria_unemployment_data['Education_Level'].isnull().sum()

np.int64(0)

In [36]:
#let's check for missing values under the region column
nigeria_unemployment_data['Region'].isnull().sum()

np.int64(250)

In [39]:
#we will fill the missing values in the region column with 'unknown' since the region is a categorical variable
#and we cannot use mean, median or mode to fill in the missing values

nigeria_unemployment_data['Region'] = nigeria_unemployment_data['Region'].fillna('Unknown')

In [40]:
#let's now confirm if Region column has no missing values anymore
nigeria_unemployment_data['Region'].isnull().sum()

np.int64(0)

In [41]:
#let's view the first 10 rows of the cleaned dataset
nigeria_unemployment_data.head(10)

,Age,Gender,Region,Location,Education_Level,Employment_Status,Years_Of_Experience,Monthly_Income_NGN
0,53,Male,North West,Urban,Secondary,Employed,37.000000,84220.840
1,43,Female,South West,Urban,Primary,Employed,24.000000,69911.720
2,29,Female,South South,Rural,Secondary,Underemployed,21.835333,31096.210
3,57,Female,South West,Rural,Primary,Employed,42.000000,108994.910
4,22,Female,South South,Rural,Secondary,Underemployed,5.000000,31807.560
5,35,Male,South East,Urban,Secondary,Employed,17.000000,42908.875
6,53,Female,North West,Urban,Primary,Underemployed,35.000000,31198.390
7,33,Male,North West,Rural,Secondary,Employed,18.000000,152089.710
8,37,Male,North East,Urban,Primary,Employed,21.000000,118601.560
9,25,Female,North West,Urban,Primary,Underemployed,10.000000,38302.580


In [42]:
#All clean. Now we will download the cleaned dataset to a new csv file called nigeria_unemployment_cleaned.csv
nigeria_unemployment_data.to_csv('nigeria_unemployment_cleaned.csv', index=False)

Next up will be outlier detection and fixing data types.